# MACS 30111 — M7 Pandas, day 3: pivot, then merge
Slides: `M7_Pandas_30111.pptx`, slides 45–58 · Textbook: §4.3.6 ("Pivoting") · Docs: https://pandas.pydata.org/docs/user_guide/merging.html

**How to use this notebook:** run the **Example** cells as we go. In each **Try it** cell, predict the answer first, then write the code. Try its marked **(on your own)** are homework.

*Packages (Anaconda): this notebook uses `openpyxl` (for `read_excel`), `plotly` and `seaborn`. If an import fails, run `conda install openpyxl plotly seaborn` once in a terminal (Anaconda Prompt on Windows), then restart the kernel.*

In [ ]:
import pandas as pd

trees = pd.read_csv("data/2015_Street_Tree_Census_sample.csv", index_col="tree_id")
trees.shape

## 1. Pivot, with and without `reset_index` (slides 45–52)
`pivot` turns a **long** table into a **wide** one. It needs three **columns**: one for the new row labels (`index=`), one for the new column names (`columns=`), and one for the values (`values=`).

Start with a count of trees for each borough and status. **Predict:** is `counts` a Series or a DataFrame? Where do borough and status live?

In [ ]:
counts = trees.groupby(["borough", "status"]).size()
print(type(counts))
counts.head()

### Example: pivot *without* `reset_index`
**Predict:** will this work? (This was quiz question 6.)

In [ ]:
counts.pivot(index="borough", columns="status")

It raises an error. `counts` is a **Series**, and borough and status are its **index** (the row labels), not columns. A Series has no `pivot`, and even if it did, there would be no `borough` *column* to name.

### Example: pivot *with* `reset_index`
`reset_index()` moves the index levels into ordinary columns and gives the rows plain labels 0, 1, 2, ... Use `name=` to name the column that holds the values.

In [ ]:
long = counts.reset_index(name="n")
print(long.shape)       # (15, 3): one row per (borough, status)
long.head()

In [ ]:
wide = long.pivot(index="borough", columns="status", values="n")
print(wide.shape)       # (5, 3)
wide

**Compare:** `counts.unstack()` gives the same wide table in one step, because it works on the index directly. `pivot` works on columns, which is why it needs `reset_index` first.

What if you leave out `name=`? The new column is named `0`, so you have to rename it (the textbook does this):

In [ ]:
counts.reset_index().columns    # ['borough', 'status', 0]

`reset_index` works on a **DataFrame** too: it moves the index (here, `tree_id`) back into a column. Remember this for merging: a merge keeps columns, not the index.

In [ ]:
trees.reset_index()[["tree_id", "borough", "spc_common"]].head(3)

### Try it 1.1
Make a wide table of **health** counts: one row per borough, one column per health value (Fair, Good, Poor). Use `groupby` → `size()` → `reset_index(name="n")` → `pivot`. What is its shape?

In [ ]:
# your code here


### Try it 1.2 (on your own)
Make a wide table of the **average** `tree_dbh` for each borough (rows) and status (columns), this time starting from `.mean()` instead of `.size()`. Which `reset_index` call do you need? *(Hint: after `groupby(...)["tree_dbh"].mean()` the values already have a name.)*

In [ ]:
# your code here


## 2. Combining DataFrames (slide 53)
| Tool | What it does | Matches rows on |
|---|---|---|
| `pd.concat([df1, df2])` | stacks tables on top of each other | nothing: rows are just stacked |
| `df1.merge(df2)` | lines up rows whose key values match | key column(s) |
| `df1.join(df2)` | a merge that uses `df2`'s index as the key | `df2`'s index |

**Rule of thumb:** concat to add more rows of the same kind; merge to add columns from a different table.

In [ ]:
queens = trees[trees["borough"] == "Queens"]
bronx = trees[trees["borough"] == "Bronx"]
both_boroughs = pd.concat([queens, bronx])
print(len(queens), len(bronx), both_boroughs.shape)   # 36,479 + 12,596 = 49,075 rows

## 3. Merge: syntax and which rows are kept (slides 54–55)
```python
left.merge(right, how="inner", on=None, left_on=None, right_on=None,
           suffixes=("_x", "_y"), indicator=False)
```
`pd.merge(left, right, ...)` does exactly the same thing.

Two small tables, both with a `name` column:

In [ ]:
grades = pd.DataFrame({"name": ["Wei", "Priya", "Mateo"], "pa1": [92, 85, 78]})
majors = pd.DataFrame({"name": ["Priya", "Mateo", "Amara"],
                       "major": ["Econ", "Sociology", "CS"]})
print(grades)
print(majors)

**Predict before each cell:** which names will be in the result?

In [ ]:
grades.merge(majors, on="name", how="inner")    # the default

In [ ]:
grades.merge(majors, on="name", how="left")

In [ ]:
grades.merge(majors, on="name", how="right")

In [ ]:
grades.merge(majors, on="name", how="outer", indicator=True)

Notice:
- A missing match becomes `NaN`, and that turns the int column `pa1` into floats (85 → 85.0).
- `indicator=True` adds a `_merge` column: `left_only`, `right_only` or `both`.
- `pd.merge(grades, majors, on="name")` gives the same result as `grades.merge(majors, on="name")`.

### Try it 3.1 (on your own)
Without running anything first, write down how many **rows** each `how=` gives (inner, left, right, outer). Then check with a loop that prints `len(...)` for each one.

In [ ]:
# your code here


### Try it 3.2 (on your own)
Both tables now also have a `term` column. Run the merge two ways: with `on="name"`, and with no `on=` at all. How many rows does each give, and what happened to the `term` columns? Why is leaving out `on=` risky?

In [ ]:
# your code here


## 4. Example: add lifespans to the trees (slide 56)
`trees_lifespan.xlsx` has one row per **species**: its common name, scientific name, and average and maximum lifespan in years.

In [ ]:
trees_life = pd.read_excel("data/trees_lifespan.xlsx")
print(trees_life.shape)
trees_life.head()

**Which column should we match on?** Compare how the two tables write a red maple:

In [ ]:
print(trees.loc[180683, ["spc_common", "spc_latin"]])
print(trees_life[trees_life["Scientific Name"] == "Acer rubrum"])

In [ ]:
# matching on the common names: "red maple" vs "Maple, Red" never line up
trees.merge(trees_life, left_on="spc_common", right_on="Common Name").shape    # only 2 trees!

The scientific names are written the same way, so we match on those. The columns have **different names**, so we use `left_on` / `right_on`. `how="left"` keeps every tree, even if its species has no lifespan.

One more thing: **merge returns plain row labels 0, 1, 2, ...**, so the `tree_id` index would be lost. Use `reset_index()` first to keep it as a column.

In [ ]:
trees_plus = trees.reset_index().merge(trees_life, how="left",
                                       left_on="spc_latin", right_on="Scientific Name",
                                       indicator=True)
print(trees_plus.shape)     # (100000, 50): every tree kept
trees_plus[["tree_id", "spc_common", "Common Name", "Average_lifespan", "_merge"]].head()

### Try it 4.1
Run the same merge **without** `reset_index()`. What happened to the tree ids? How would you look up tree 180683 now?

In [ ]:
# your code here


## 5. Always check your merge (slide 57)
Compare the shape before and after, and look at the `_merge` counts.

In [ ]:
trees_plus["_merge"].value_counts()      # left_only 80,884 · both 19,116

### Try it 5.1 (on your own)
Most trees found no lifespan. Which **5 species** (by `spc_latin`) have the most unmatched trees?

In [ ]:
# your code here


### Try it 5.2 (on your own)
The lifespan file spells green ash *Fraxinus pennsylvanicum*; the census spells it *Fraxinus pennsylvanica*. Fix the spelling in `trees_life` with `.replace`, merge again, and count the matched trees. How many more trees found a lifespan?

In [ ]:
# your code here


## 6. TASK: ask a question with the merged data (slide 58)

### Try it 6.1 (on your own)
Which borough's street trees have the **longest average lifespan**? *(Hint: `groupby` after the merge. What happens to the trees with no lifespan?)*

In [ ]:
# your code here


### Try it 6.2 (on your own)
Among the trees that matched, which **species** is planted most often, and which matched species has the **longest** average lifespan?

In [ ]:
# your code here


## 7. Three tables: merge practice (handout, front)
Trees from our sample and lifespans from `trees_lifespan.xlsx`. **Predict on the handout first**, then run each prompt.

In [ ]:
queens = pd.DataFrame({"tree_id": [180683, 196765, 203326],
                       "borough": ["Queens", "Queens", "Queens"],
                       "spc_latin": ["Acer rubrum", "Quercus palustris", "Gleditsia triacanthos"],
                       "spc_common": ["red maple", "pin oak", "honeylocust"],
                       "dbh": [3, 20, 10]})
brooklyn = pd.DataFrame({"tree_id": [180707, 196132],
                         "borough": ["Brooklyn", "Brooklyn"],
                         "spc_latin": ["Quercus palustris", "Platanus x acerifolia"],
                         "spc_common": ["pin oak", "London planetree"],
                         "dbh": [17, 19]})
life = pd.DataFrame({"Scientific Name": ["Acer rubrum", "Quercus palustris", "Fagus grandifolia"],
                     "Average_lifespan": [130, 100, 300]})

### Prompt 1

In [ ]:
pd.concat([queens, brooklyn])

### Prompt 2

In [ ]:
queens.merge(life, left_on="spc_latin", right_on="Scientific Name")

### Prompt 3

In [ ]:
queens.merge(life, left_on="spc_latin", right_on="Scientific Name", how="left")

### Prompt 4

In [ ]:
all_trees = pd.concat([queens, brooklyn])
all_trees.merge(life, left_on="spc_latin", right_on="Scientific Name",
                how="outer", indicator=True)

### Prompt 5

In [ ]:
queens.merge(life)

In [ ]:
queens.merge(brooklyn)     # no error... but look at the shape

### Prompt 6

In [ ]:
queens.join(life.set_index("Scientific Name"), on="spc_latin")

### Prompt 7 (discuss)
Why do we care about this pre-work (predicting the rows, checking the keys) instead of just letting Python do the merge? *Hint: which of prompts 1–6 gave an error, and which ran without complaint but gave something you might not expect?*

## 8. Plotting from a table (handout, back; `M7_Pandas_30111_merge_plots.pptx` slides 13–20)
**The shape of your table decides how you plot it.**
- **Wide** (one row per x position, one column per series): pandas `.plot` / matplotlib draws every column for you.
- **Long** (one row per bar or point): plotly and seaborn want you to name the columns for x, y and color.

*If `import plotly.express` fails, run `conda install plotly` once in a terminal (Anaconda Prompt on Windows) and restart the kernel. Seaborn: `conda install seaborn`.*

In [ ]:
import matplotlib.pyplot as plt
import plotly.express as px

def tree_health_by_boro(trees):
    combined_status = trees.status.where(trees.status != "Alive", trees.health)
    num_per_boro = trees.groupby("borough").size()
    combined_per_boro = trees.groupby(["borough", combined_status]).size()
    pct_per_boro = combined_per_boro / num_per_boro * 100.0
    pct_per_boro_df = pct_per_boro.unstack()
    return pct_per_boro_df[["Good", "Fair", "Poor", "Dead", "Stump"]]

wide = tree_health_by_boro(trees).round(1)
wide       # wide: one row per borough, one column per status

### Example 1a: wide table → pandas `.plot` (matplotlib underneath) (slide 15)

In [ ]:
ax = wide.plot.bar(rot=0)
ax.set_ylabel("percent of trees")
ax.set_title("Tree health by borough");

### Example 1b: the same data, long → plotly (slide 18)
`melt` turns the wide table into one row per (borough, status) bar.

In [ ]:
long = wide.reset_index().melt(id_vars="borough", var_name="status", value_name="percent")
print(long.shape)     # (25, 3)
px.bar(long, x="borough", y="percent", color="status", barmode="group",
       title="Tree health by borough")

### Example 1c: plain matplotlib (slide 16)
Drawing grouped bars by hand takes a loop: one `ax.bar` call per status, each shifted a little to the right. *(The slide's version sets the tick labels without setting the tick positions; `set_xticks` fixes that.)*

In [ ]:
import numpy as np

fig, ax = plt.subplots(figsize=(8, 4))
bar_width = 0.15
x = np.arange(len(wide))                          # one position per borough
for i, status in enumerate(wide.columns):
    ax.bar(x + i * bar_width, wide[status], bar_width, label=status)
ax.set_xticks(x + 2 * bar_width)
ax.set_xticklabels(wide.index)
ax.set_xlabel("Borough")
ax.set_ylabel("percent of trees")
ax.set_title("Tree health by borough")
ax.legend(title="status")
plt.tight_layout()

### Example 1d: seaborn wants long too (slide 17)

In [ ]:
import seaborn as sns

sns.barplot(long, x="borough", y="percent", hue="status");

### Example 2: counts, stacked
Here we start **long** (from `groupby` + `reset_index`) and `pivot` to wide for matplotlib.

In [ ]:
counts_long = trees.groupby(["borough", "status"]).size().reset_index(name="n")
counts_wide = counts_long.pivot(index="borough", columns="status", values="n")

ax = counts_wide.plot.bar(stacked=True, rot=0)       # wide -> matplotlib
ax.set_ylabel("number of trees");

In [ ]:
px.bar(counts_long, x="borough", y="n", color="status")    # long -> plotly (stacks by default)

### The mix and match (handout, back)
Tables **A–D** from the handout, built from the tables above. Do the handout on paper first; then run each plot below to check your letters. Every plot calls its table `df`, exactly as on the handout.

In [ ]:
import seaborn as sns

A = counts_wide        # wide: 5 x 3, one row per borough, one column per status
B = long               # long: 25 x 3, borough, status, percent
C = wide               # wide: 5 x 5, one row per borough, one column per health status
D = counts_long        # long: 15 x 3, borough, status, n

#### Plot 1: table D (long)
plotly names a column for x, y and color, so it needs a **long** table. `y="n"` points to D. plotly stacks by default.

In [ ]:
df = D    # counts_long
px.bar(df, x="borough", y="n", color="status")

#### Plot 2: table C (wide)
pandas `.plot` draws one bar color per **column**, so it needs a **wide** table. Percents, side by side: C.

In [ ]:
df = C    # wide
df.plot.bar(rot=0)

#### Plot 3: table B (long)
seaborn, like plotly, names columns, so it needs **long**. Here x is status and the colors are boroughs: the same table B, just different roles.

In [ ]:
df = B    # long
sns.barplot(df, x="status", y="percent", hue="borough")

#### Plot 4: table A (wide)
`.plot` again (wide), but counts and stacked: A.

In [ ]:
df = A    # counts_wide
df.plot.bar(stacked=True, rot=0)

#### Plot 5: table B (long)
plotly (long), `y="percent"`, side by side: B again.

In [ ]:
df = B    # long
px.bar(df, x="borough", y="percent", color="status", barmode="group")

### Try it 8.1
Plot the **health counts** from Try it 1.1 (Fair / Good / Poor by borough) two ways: from the wide table with `.plot.bar()`, and from the long table with `px.bar`. Which table did each one need?

In [ ]:
# your code here


### Try it 8.2 (on your own)
Using the merged `trees_plus` from §4, plot the **average lifespan by borough** as a bar chart. Is this table long or wide? Then try `px.bar` with `color="borough"`.

In [ ]:
# your code here


### Bonus (on your own): the same five plots in plain matplotlib
pandas `.plot`, plotly and seaborn all sit on top of something lower level. Here you draw each handout plot with plain matplotlib (`ax.bar`), like slide 16. You'll need a **wide** table and a loop: one `ax.bar` call per color, shifted (side by side) or stacked with `bottom=`. Compare how much code each one takes.

*Hints:* `fig, ax = plt.subplots()` · `x = np.arange(len(table))` · side by side: `ax.bar(x + i * w, values, w, label=...)` then `ax.set_xticks(x + 2 * w, labels)` · stacked: `ax.bar(labels, values, bottom=bottom)` and add each column to `bottom`.

### Bonus 8.3 (on your own): Plot 1 (stacked counts, from D)

In [ ]:
# your code here


### Bonus 8.4 (on your own): Plot 2 (grouped percents, from C)

In [ ]:
# your code here


### Bonus 8.5 (on your own): Plot 3 (x = status, color = borough, from B)

In [ ]:
# your code here


### Bonus 8.6 (on your own): Plot 4 (stacked counts, from A)

In [ ]:
# your code here


### Bonus 8.7 (on your own): Plot 5 (grouped percents, from B)

In [ ]:
# your code here


## 9. On your own: creating new variables from old (`M7_Pandas_30111_merge_plots.pptx` slides 21–24)
Often you have a continuous-ish column (a lifespan, a diameter) and you want a **category** you can group or plot by, or a **recode** of an existing column. Three tools here; §10 adds three bonus ones. Every Try it in §9 and §10 is **on your own**.

We'll use the lifespan table (`trees_life`, from §4) and `trees`.

### Categorical (slide 21)
A text column with only a few different values can be stored as a **category**. pandas keeps each value once and stores small codes for the rows, which saves memory and keeps a fixed order.

In [ ]:
import numpy as np

cats = trees[["borough", "status", "health"]].copy()
before = cats.memory_usage(deep=True).sum()
for col_name in ["borough", "status", "health"]:
    cats[col_name] = cats[col_name].astype("category")
after = cats.memory_usage(deep=True).sum()
print(before, after)                       # bytes: the category version is several times smaller
cats["borough"].cat.categories

### Try it 9.1 (on your own)
Convert `spc_common` to a category. How many categories does it have? How does that compare with `trees["spc_common"].nunique()`?

In [ ]:
# your code here


### `pd.cut` and `pd.qcut` (slides 23–24)
- `pd.cut(..., bins=4)` cuts the **range** into 4 equal-width slices.
- `pd.qcut(..., q=4)` cuts the **distribution** into 4 slices with about the same number of rows each.
- `pd.cut(..., bins=[...])` lets you choose the cut points yourself.

In [ ]:
labels = ["Short", "Medium", "Long", "ExtraLong"]
trees_life["lifespan_cat_split"] = pd.cut(trees_life["Average_lifespan"], bins=4, labels=labels)
trees_life["lifespan_cat_eq"] = pd.qcut(trees_life["Average_lifespan"], q=4, labels=labels)
trees_life["lifespan_cat"] = pd.cut(trees_life["Average_lifespan"],
                                    bins=[0, 90, 200, 300, 600], labels=labels)
trees_life[["Common Name", "Average_lifespan", "lifespan_cat_split",
            "lifespan_cat_eq", "lifespan_cat"]].head()

### Try it 9.2 (on your own)
Count the species in each category for all three columns (`value_counts(sort=False)` keeps the Short → ExtraLong order). Why does `bins=4` put almost everything in Short and Medium? Why isn't `qcut` exactly 15 / 15 / 15 / 16? *(Hint: look at `pd.cut(trees_life["Average_lifespan"], bins=4).cat.categories`.)*

In [ ]:
# your code here


### `where` and `np.where` (slide 22)
- `series.where(cond, other)` **keeps** values where `cond` is True and **replaces** the rest with `other`.
- `np.where(cond, a, b)` builds a brand-new column: `a` where `cond` is True, `b` where it's False.

In [ ]:
cond = trees_life["Average_lifespan"] > 150
trees_life["lifespan_short"] = trees_life["Average_lifespan"].where(cond, other="short lived")
print(trees_life["lifespan_short"].value_counts().head())   # numbers kept, others replaced

trees_life["lived"] = np.where(cond, "long lived", "short lived")
trees_life["lived"].value_counts()

### Try it 9.3 (on your own)
1. Why does `lifespan_short` mix numbers and text, while `lived` has only text?
2. Use `np.where` on `trees` to make a column `big` that is `"big"` when `tree_dbh` is at least 20 and `"not big"` otherwise. How many big trees are there?

In [ ]:
# your code here


## 10. Bonus (on your own): `apply`, `map` and `lambda` (slides 25–27)
These build new columns with **your own rule**, one value at a time.

### `apply` (slide 25)
`series.apply(f)` calls your function `f` on each value. Use it when the rule is more than a single comparison.

In [ ]:
def lifespan_label(years):
    if years >= 300:
        return "very long"
    elif years >= 100:
        return "long"
    else:
        return "short"

trees_life["label"] = trees_life["Average_lifespan"].apply(lifespan_label)
trees_life["label"].value_counts()

### Try it 10.1 (bonus, on your own)
Write a function that turns a lifespan into its **century** as text (for example 120 → `"100s"`, 260 → `"200s"`, 50 → `"0s"`). Use `apply` to add a `century` column, then count the species in each century.

In [ ]:
# your code here


### `map` (slide 26)
`series.map(d)` looks up each value in a **dictionary** `d` and returns the matching value. Values not in the dictionary become `NaN`.

In [ ]:
abbrev = {"Bronx": "BX", "Brooklyn": "BK", "Manhattan": "MN",
          "Queens": "QN", "Staten Island": "SI"}
trees["boro_code"] = trees["borough"].map(abbrev)
trees["boro_code"].value_counts()

### Try it 10.2 (bonus, on your own)
Use `map` to turn `status` into a number: Alive → 1, Dead → 0, Stump → 0. What is the mean of the new column, and what does that number mean?

In [ ]:
# your code here


### `lambda` functions (slide 27)
A `lambda` is a one-line function with no name, handy inside `apply` when the rule is short:
`lambda x: <expression using x>` means the same as `def f(x): return <expression>`.

In [ ]:
trees_life["Average_lifespan"].apply(lambda years: "centenarian" if years >= 100 else "under 100").value_counts()

### Try it 10.3 (bonus, on your own)
Rewrite your `century` function from Try it 10.1 as a `lambda` inside `apply`. Check that it gives the same column.

In [ ]:
# your code here


## 11. Optional extra practice: predict the merge (cards)
*Optional; not part of today's handout.* These eight small cases come from a printed card set. For each card, **predict on paper first** (rows, columns, and the card's question), then run the card's cell and check.

### Card 1: The default

In [ ]:
grades = pd.DataFrame({"name": ["Wei", "Priya", "Mateo"], "pa1": [92, 85, 78]})
majors = pd.DataFrame({"name": ["Priya", "Mateo", "Amara"], "major": ["Econ", "Sociology", "CS"]})

# predict first, then run:
grades.merge(majors, on="name")

### Card 2: Keep everyone on the right

In [ ]:
grades = pd.DataFrame({"name": ["Wei", "Priya", "Mateo"], "pa1": [92, 85, 78]})
majors = pd.DataFrame({"name": ["Priya", "Mateo", "Amara"], "major": ["Econ", "Sociology", "CS"]})

# predict first, then run:
grades.merge(majors, on="name", how="right")

### Card 3: Count the matches

In [ ]:
roster = pd.DataFrame({"name": ["Wei", "Priya", "Mateo", "Kofi"], "section": [1, 1, 2, 2]})
attendance = pd.DataFrame({"name": ["Priya", "Kofi", "Lina"], "days": [9, 7, 10]})

# predict first, then run:
roster.merge(attendance, on="name", how="outer", indicator=True)

### Card 4: A repeated key

In [ ]:
submissions = pd.DataFrame({"name": ["Priya", "Priya", "Mateo"], "pset": [1, 2, 1]})
majors = pd.DataFrame({"name": ["Priya", "Mateo", "Amara"], "major": ["Econ", "Sociology", "CS"]})

# predict first, then run:
submissions.merge(majors, on="name")

### Card 5: Repeated on both sides

In [ ]:
submissions = pd.DataFrame({"name": ["Priya", "Priya", "Mateo"], "pset": [1, 2, 1]})
office_hours = pd.DataFrame({"name": ["Priya", "Priya", "Mateo"], "day": ["Mon", "Wed", "Fri"]})

# predict first, then run:
submissions.merge(office_hours, on="name")

### Card 6: Forgetting on=

In [ ]:
grades = pd.DataFrame({"name": ["Wei", "Priya", "Mateo"], "term": ["Fall", "Fall", "Fall"], "pa1": [92, 85, 78]})
majors = pd.DataFrame({"name": ["Priya", "Mateo", "Amara"], "term": ["Fall", "Winter", "Fall"], "major": ["Econ", "Sociology", "CS"]})

# predict first, then run:
grades.merge(majors)

### Card 7: Different key names

In [ ]:
grades = pd.DataFrame({"name": ["Wei", "Priya", "Mateo"], "pa1": [92, 85, 78]})
majors = pd.DataFrame({"student": ["Priya", "Mateo", "Amara"], "major": ["Econ", "Sociology", "CS"]})

# predict first, then run:
grades.merge(majors, left_on="name", right_on="student")

### Card 8: Almost the same key

In [ ]:
grades = pd.DataFrame({"name": ["Wei", "Priya", "Mateo"], "pa1": [92, 85, 78]})
majors = pd.DataFrame({"name": ["priya", "mateo", "Amara"], "major": ["Econ", "Sociology", "CS"]})

# predict first, then run:
grades.merge(majors, on="name")

## Next time
More plotting. Homework: Try its 1.1, 4.1 and 8.1 (if we didn't finish them in class); the **on your own** Try its (1.2, 3.1, 3.2, 5.1, 5.2, 6.1, 6.2, 8.2); the matplotlib bonus (8.3–8.7); all of §9; and the bonus §10. The cards in §11 are optional extra practice.